# EquiFockNet

This repository contains the code and Jupyter Notebooks for predicting symmetric, physically accurate Fock matrices using $SO(3)$ equivariant neural networks. It includes a customized, natively integrated version of PhiSNet.

## 1. Project Setup

This guide will walk you through setting up the environment, getting the data, and running the project. The repository is designed to be fully self-contained.

### 1.1. Code Installation

Clone the repository to your local machine:

```bash
git clone [https://github.com/marquetand/equifocknet.git](https://github.com/marquetand/equifocknet.git)
cd equifocknet
```

### 1.2. Directory Structure

The repository uses an automated directory structure to keep code, data, and outputs organized. When you run the notebook for the first time, it will automatically generate the following ignored folders:

* `data_electrocyclic/`: Place your raw molecular data here. It will also hold the compiled SQLite and `.npz` datasets.
* `training_output/`: Contains all model checkpoints, TensorBoard logs, and `.pt` model weights.
* `analysis/`: Contains all generated visualization plots (e.g., `.png` files).
* `phisnet_fork/`: The customized core neural network architecture.

### 1.3. Data Preparation

**[Note: Data source to be specified]**

You will need to download the project dataset containing the Molcas calculations. Once downloaded, extract the files directly into the `data_electrocyclic/open/` directory.

### 1.4. Environment Setup

It is highly recommended to use Conda to manage the Python environment. If you do not have Conda installed on your machine, please download and install [Miniconda](https://docs.anaconda.com/free/miniconda/index.html) first.

**1. Create and activate a new Conda environment:**
```bash
conda create -n equifocknet python=3.13
conda activate equifocknet
```

**2. Install `uv` and project dependencies:**
We use `uv`, an extremely fast Python package installer, to handle all dependencies (including PyTorch). First, install `uv`:
```bash
pip install uv
```

Next, install the project requirements. **Please choose the correct command below depending on your hardware:**

* **Option A: You have an NVIDIA GPU (Recommended)**
  This command points `uv` to the PyTorch index that contains the hardware-accelerated CUDA 11.8 bindings:
  ```bash
  uv pip install -r requirements.txt --extra-index-url [https://download.pytorch.org/whl/cu118](https://download.pytorch.org/whl/cu118)
  ```

* **Option B: You DO NOT have an NVIDIA GPU (CPU Only or macOS)**
  This command points `uv` to the default CPU-only PyTorch index:
  ```bash
  uv pip install -r requirements.txt --extra-index-url [https://download.pytorch.org/whl/cpu](https://download.pytorch.org/whl/cpu)
  ```

## 2. Running the Model

Once your environment is set up and the data is placed in `data_electrocyclic/open/`, you can launch the Jupyter Notebook:

```bash
jupyter notebook equifocknet.ipynb
```

The notebook is configured to automatically route all data reading and model saving to the correct internal folders. You can seamlessly switch between overfitting tests and full production runs by adjusting the `N_TRAIN_SAMPLES` variable in the first cell of the notebook.

In [ ]:
# --- CELL 1: GLOBAL IMPORTS, SETUP, AND DATA LOADING ---
import os
import sys
import copy
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.utils.data import Dataset, DataLoader
import pytorch_lightning as pl
from scipy.spatial.transform import Rotation
import ase.io
import ase.db
import ase.db.sqlite
import logging
import warnings
from IPython.display import clear_output
from ase.data import chemical_symbols

# PhiSNet specific imports
from phisnet_fork.nn.neural_network import NeuralNetwork as CorePhiSNet
from phisnet_fork.utils.transform_hamiltonians import (
    transform_hamiltonians_from_ao_to_lm,
    transform_hamiltonians_from_lm_to_ao
)

from phisnet_fork.utils.definitions import orbital_definitions

# Dynamically define and create output directories
BASE_DIR = os.path.abspath("")
DATA_DIR = os.path.join(BASE_DIR, "data_electrocyclic")
OUT_DIR = os.path.join(BASE_DIR, "training_output")
ANALYSIS_DIR = os.path.join(BASE_DIR, "analysis")

if not os.path.exists(DATA_DIR):
    print(f"⚠️ WARNING: The data directory '{DATA_DIR}' was not found!")
    print("Please ensure you have downloaded and extracted the dataset into this folder before running the next cells.")
else:
    print(f"✅ Data directory found.")

os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(ANALYSIS_DIR, exist_ok=True)


# --- BULLETPROOF ASE MONKEY PATCH ---
_old_metadata_prop = ase.db.sqlite.SQLite3Database.metadata

def _patched_metadata_getter(self):
    if self._metadata is None:
        if getattr(self, "connection", None) is None:
            self.connection = self._connect()
        if not getattr(self, "_initialized", False):
            self._initialize(self.connection)
        if not hasattr(self, "change_count"):
            self.change_count = 0
    return _old_metadata_prop.fget(self)

ase.db.sqlite.SQLite3Database.metadata = property(
    fget=_patched_metadata_getter,
    fset=_old_metadata_prop.fset
)
# ------------------------------------

# --- MASTER CONFIGURATION ---
# Fixed the line below where base_name was previously hidden in a comment
data_directory = os.path.join(DATA_DIR, "open") 
base_name = "open_g2m05_"

convention = "molcas_ANO-VDZP"
fock_subfolder = "VDZP"          # ← Subdirectory containing fock_real.npy (enter "MB" for MB data)

# nbasis = 32

# 1. How many structures to load into the database? (Do this once)
N_TOTAL_STRUCTURES = 10 

# 2. THE SWITCH: How many samples to actually train on? 
# Set to 1 for overfitting, or up to 800 for full training (leaving 200 for val/test)
N_TRAIN_SAMPLES = 8

print(f"Loading {N_TOTAL_STRUCTURES} structures from {data_directory}...")

structures = []
fock_matrices = []

# Verify path exists before looping to avoid obscure FileNotFoundErrors
if os.path.exists(data_directory):
    for n in range(1, N_TOTAL_STRUCTURES + 1):
        ns = str(n).zfill(4)
        xyz_filename = os.path.join(data_directory, base_name + ns, base_name + ns + '.xyz')
        structures.append(ase.io.read(xyz_filename, format="xyz"))
        
        npy_filename = os.path.join(data_directory, base_name + ns, fock_subfolder, 'fock_real.npy')
        fock_matrices.append({"fock_matrix": np.load(npy_filename).flatten()})
    print("All files loaded into memory successfully!")
else:
    print(f"❌ ERROR: Subdirectory '{data_directory}' not found. Check your data extraction.")

In [ ]:
# --- CELL 2: BUILD ASE AND PHISNET DATABASES ---
db_path = os.path.join(DATA_DIR, "open.db")
phisnet_npz_path = os.path.join(DATA_DIR, "phisnet_dataset.npz")

# 1. Build ASE Database
if os.path.exists(db_path):
    os.remove(db_path)
    print(f"Deleted old {db_path} to start fresh.")

print(f"Writing structures to {db_path}...")
with ase.db.connect(db_path) as conn:
    for idx in range(len(structures)):
        conn.write(structures[idx], data=fock_matrices[idx], key_value_pairs={"idx": idx})
    conn.metadata = {
        "_distance_unit": 'Ang',
        "_property_unit_dict": {'fock_matrix': 1.0}
    }

# 2. Build PhiSNet Dataset (AO to LM Transformation)
all_Z, all_R, all_F = [], [], []

print("Converting ASE database to PhiSNet spherical harmonic format...")
with ase.db.connect(db_path) as conn:
    for i in range(1, conn.count() + 1):
        row = conn.get(i)
        atoms = row.toatoms()
        F_ao = row.data['fock_matrix']
        
        n_basis = int(np.sqrt(len(F_ao)))
        F_ao = F_ao.reshape(1, n_basis, n_basis)
        F_lm = transform_hamiltonians_from_ao_to_lm(F_ao, atoms=list(atoms.symbols), convention=convention)[0]
        
        all_Z.append(atoms.numbers)
        all_R.append(atoms.positions)
        all_F.append(F_lm)
        
np.savez(phisnet_npz_path, Z=np.array(all_Z), R=np.array(all_R), F=np.array(all_F))
print(f"Dataset successfully saved to {phisnet_npz_path}!")

In [ ]:
# --- CELL 3: DYNAMIC SPLIT AND DATALOADERS ---
import pytorch_lightning as pl

# 1. Deterministic splitting
pl.seed_everything(29, workers=True)

# 2. Get actual dataset length instead of hardcoding N_TOTAL_STRUCTURES
dataset_data = np.load(phisnet_npz_path)
total_samples = len(dataset_data['Z'])
indices = np.arange(total_samples)
np.random.shuffle(indices)

# Dynamic Split Logic
if N_TRAIN_SAMPLES == 1:
    print("MODE: Overfitting Test (1 Sample)")
    train_idx, val_idx, test_idx = indices[:1], indices[:1], indices[:1]
    batch_size = 1
else:
    print(f"MODE: Standard Training ({N_TRAIN_SAMPLES} Samples)")
    val_size = max(1, int(0.1 * total_samples))
    test_size = max(1, int(0.1 * total_samples))
    actual_train = min(N_TRAIN_SAMPLES, total_samples - val_size - test_size)
    
    train_idx = indices[:actual_train]
    val_idx = indices[actual_train : actual_train + val_size]
    test_idx = indices[actual_train + val_size : actual_train + val_size + test_size]
    batch_size = 16

split_file = os.path.join(OUT_DIR, f"phisnet_split_{N_TRAIN_SAMPLES}.npz")
np.savez(split_file, train_idx=train_idx, val_idx=val_idx, test_idx=test_idx)

class PhiSNetDataset(Dataset):
    def __init__(self, npz_file, indices):
        data = np.load(npz_file)
        self.Z = data['Z'][indices]
        self.R = data['R'][indices]
        self.F = data['F'][indices]
        
    def __len__(self):
        return len(self.Z)

    def __getitem__(self, idx):
        # UPGRADE: Return float64 natively to satisfy physics requirements without casting overhead
        return {
            'Z': torch.tensor(self.Z[idx], dtype=torch.long),
            'R': torch.tensor(self.R[idx], dtype=torch.float64),
            'F': torch.tensor(self.F[idx], dtype=torch.float64)
        }

def collate_fn(batch):
    return {
        'Z': torch.stack([item['Z'] for item in batch]),
        'R': torch.stack([item['R'] for item in batch]),
        'F': torch.stack([item['F'] for item in batch])
    }

train_loader = DataLoader(PhiSNetDataset(phisnet_npz_path, train_idx), batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(PhiSNetDataset(phisnet_npz_path, val_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)
test_loader = DataLoader(PhiSNetDataset(phisnet_npz_path, test_idx), batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"DataLoaders ready! Train: {len(train_idx)} | Val: {len(val_idx)} | Test: {len(test_idx)}")

In [ ]:
# --- CELL 4: Transform database ---

# Import the utilities from your fork
from phisnet_fork.utils.transform_hamiltonians import (
    transform_hamiltonians_from_ao_to_lm,
    transform_hamiltonians_from_lm_to_ao
)

def test_transformation(convention, db_path):
    
    # 1. Load a single matrix and geometry from your database
    print("Loading test molecule...")
    with ase.db.connect(db_path) as conn:
        row = conn.get(1) # Get the very first molecule
        target_flat = row.data['fock_matrix']
        atoms = row.toatoms()
        
    atomic_symbols = list(atoms.symbols)
    n_basis = int(np.sqrt(len(target_flat)))
    F_original = target_flat.reshape(1, n_basis, n_basis) # Add batch dimension
    
    print(f"Original Matrix Shape: {F_original.shape}")
    print(f"Atoms: {''.join(atomic_symbols)}")
    
    # 2. Forward Transform: Atomic Orbitals (AO) -> Spherical Harmonics (LM)
    try:
        F_lm = transform_hamiltonians_from_ao_to_lm(
            F_original, 
            atoms=atomic_symbols, 
            convention=convention
        )
        print(f"Successfully transformed to LM. New Shape: {F_lm.shape}")
    except Exception as e:
        print(f"FAILED on Forward Transform: {e}")
        sys.exit(1)
        
    # 3. Reverse Transform: Spherical Harmonics (LM) -> Atomic Orbitals (AO)
    try:
        F_reconstructed = transform_hamiltonians_from_lm_to_ao(
            F_lm, 
            atoms=atomic_symbols, 
            convention=convention
        )
        print(f"Successfully transformed back to AO. Reconstructed Shape: {F_reconstructed.shape}")
    except Exception as e:
        print(f"FAILED on Reverse Transform: {e}")
        sys.exit(1)
        
    # 4. The Moment of Truth: Calculate the Error
    max_error = np.max(np.abs(F_original - F_reconstructed))
    mean_error = np.mean(np.abs(F_original - F_reconstructed))
    
    print("-" * 30)
    print(f"Max Absolute Error:  {max_error:.8e}")
    print(f"Mean Absolute Error: {mean_error:.8e}")
    
    if max_error < 1e-10:
        print("\nSUCCESS! The transformation dictionary is mathematically perfect. We are ready to build the dataset!")
    else:
        print("\nWARNING: The transformation introduced errors. Do not proceed to training until we fix the dictionary.")

if __name__ == "__main__":
    test_transformation(convention=convention, db_path=db_path)


In [ ]:
# --- CELL 5: PHISNET PYTORCH LIGHTNING TRAINING LOOP ---

logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
warnings.filterwarnings("ignore")

class LivePlotCallback(pl.Callback):
    def __init__(self):
        super().__init__()
        self.train_losses, self.val_losses = [], []
        self.train_epochs, self.val_epochs = [], []

    def on_train_epoch_end(self, trainer, pl_module):
        metrics = trainer.callback_metrics
        if 'train_loss' in metrics and 'val_loss' in metrics:
            self.train_losses.append(metrics['train_loss'].item())
            self.val_losses.append(metrics['val_loss'].item())
            
            self.train_epochs.append(trainer.current_epoch)
            # Shifted to perfectly align the weights mathematically
            self.val_epochs.append(trainer.current_epoch + 1)
            
            clear_output(wait=True)
            plt.figure(figsize=(10, 5))
            plt.plot(self.train_epochs, self.train_losses, label='Train Loss (Before Update)', color='#1f77b4', marker='o', linewidth=2)
            plt.plot(self.val_epochs, self.val_losses, label='Validation Loss (After Update)', color='#ff7f0e', marker='s', linewidth=2)
            plt.xlabel('Epoch', fontsize=12)
            plt.ylabel('Loss (MSE)', fontsize=12)
            plt.title('Live Training vs Validation Error (Shift Corrected)', fontsize=14)
            plt.yscale('log')
            plt.grid(True, which="both", ls="--", alpha=0.5)
            plt.legend(fontsize=12)
            plt.show()

class PhisNetLightning(pl.LightningModule):
    # FIXED: Restored the aggressive 1e-3 learning rate for much faster convergence
    def __init__(self, model, learning_rate=1e-3): 
        super().__init__()
        self.model = model
        self.learning_rate = learning_rate
        self.loss_key = 'full_hamiltonian'

    def forward(self, R):
        return self.model(R)

    def loss_fn(self, pred, targets):
        # We keep the vectorized MSE for blazing fast speeds
        return torch.nn.functional.mse_loss(pred[self.loss_key], targets['F'])

    def training_step(self, batch, batch_idx):
        pred = self(batch['R'])
        loss = self.loss_fn(pred, batch)
        self.log("train_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        pred = self(batch['R'])
        loss = self.loss_fn(pred, batch)
        self.log("val_loss", loss, on_step=False, on_epoch=True, prog_bar=True)
        return loss

    def configure_optimizers(self):
        optimizer = torch.optim.AdamW(self.parameters(), lr=self.learning_rate, weight_decay=0.0, amsgrad=True)
        
        # FIXED: Switched to 'rel' mode. 
        # Now it looks for a 0.01% (1e-4) relative improvement, regardless of the absolute scale of the MSE!
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, mode='min', factor=0.1, patience=5, threshold=1e-4, threshold_mode='rel'
        )
        return {"optimizer": optimizer, "lr_scheduler": {"scheduler": scheduler, "monitor": "val_loss"}}

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
sample_Z = train_loader.dataset[0]['Z'].cpu().numpy()

# Build the (Z, l) orbital spec automatically from the convention's definition,
# so it always stays in sync with the basis set (no manual per-basis editing).
def build_orbital_map(convention):
    """Z -> [(Z, l), ...] derived from orbital_definitions[convention]."""
    defs = orbital_definitions[convention]
    return {int(Z): [(int(Z), int(l)) for l in l_list] for Z, l_list in defs.items()}

orbital_map = build_orbital_map(convention)
molecule_orbitals = [orbital_map[z] for z in sample_Z]

# Maximum orbital angular momentum present in this basis (s=0, p=1, d=2, ...),
# computed exactly the way the model does internally. The network needs order
# >= 2*L_max to represent the Hamiltonian blocks without loss (Clebsch-Gordan
# coupling of two L orbitals produces features up to 2*L).
L_max = max(l for atom_orbitals in molecule_orbitals for (z, l) in atom_orbitals)
order = 2 * L_max

core_model = CorePhiSNet(
    orbitals=molecule_orbitals, 
    num_features=128,
    order=order,         
    num_modules=5 
).double()

core_model.calculate_full_hamiltonian = True    # The only needed
core_model.calculate_core_hamiltonian = False
core_model.calculate_overlap_matrix   = False
core_model.calculate_energy           = False
core_model.calculate_forces           = False
core_model.create_graph               = False

# Ensure the class gets instantiated with the restored learning rate
lightning_model = PhisNetLightning(core_model, learning_rate=1e-3)

logger = pl.loggers.TensorBoardLogger(save_dir=OUT_DIR, name=f'phisnet_train_log_{N_TRAIN_SAMPLES}')
checkpoint_callback = pl.callbacks.ModelCheckpoint(
    dirpath=OUT_DIR, filename=f"phisnet_best_model_{N_TRAIN_SAMPLES}",
    save_top_k=1, monitor="val_loss", mode="min"
)

trainer = pl.Trainer(
    callbacks=[checkpoint_callback, LivePlotCallback()],
    logger=logger, max_epochs=100,
    accelerator="gpu" if torch.cuda.is_available() else "cpu", devices=1,
    enable_model_summary=False, num_sanity_val_steps=0
)

print(f"Starting Training on {N_TRAIN_SAMPLES} samples...")
trainer.fit(lightning_model, train_dataloaders=train_loader, val_dataloaders=val_loader)
torch.save(lightning_model.model, os.path.join(OUT_DIR, f"phisnet_raw_{N_TRAIN_SAMPLES}.pt"))
print("Training Complete!")

In [ ]:
# --- CELL 6: COMPREHENSIVE SET EVALUATION ---

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# TRY TO LOAD THE BEST CHECKPOINT FIRST
# Lightning appends '-v1.ckpt' or similar if multiple runs happen, 
# but usually it's just .ckpt
best_model_path = os.path.join(OUT_DIR, f"phisnet_best_model_{N_TRAIN_SAMPLES}.ckpt")
last_model_path = os.path.join(OUT_DIR, f"phisnet_raw_{N_TRAIN_SAMPLES}.pt")

if os.path.exists(best_model_path):
    print(f"Loading BEST model from checkpoint: {best_model_path}")
    # We use PhisNetLightning.load_from_checkpoint to get the model out of the Lightning wrapper
    trained_wrapper = PhisNetLightning.load_from_checkpoint(best_model_path, model=core_model)
    model = trained_wrapper.model.to(device)
elif os.path.exists(last_model_path):
    print(f"⚠️ Best checkpoint not found. Loading LAST saved model instead: {last_model_path}")
    model = torch.load(last_model_path, map_location=device, weights_only=False).to(device)
else:
    print(f"❌ ERROR: No trained model found in {OUT_DIR}!")
model.eval()

# Assuming the same molecule across the dataset (standard for single-trajectory dynamics)
sample_Z = train_loader.dataset[0]['Z'].cpu().numpy()
atomic_symbols = [chemical_symbols[z] for z in sample_Z]

def evaluate_loader(loader, split_name):
    true_ao_list = []
    pred_ao_list = []
    
    for batch in loader:
        R = batch['R'].to(device).double()
        target_F_lm = batch['F'].numpy() 
        
        # Pass ONLY the positions (R) to the raw model
        pred_F_lm = model(R)['full_hamiltonian'].cpu().numpy()
        
        pred_F_ao = transform_hamiltonians_from_lm_to_ao(pred_F_lm, atomic_symbols, convention)
        target_F_ao = transform_hamiltonians_from_lm_to_ao(target_F_lm, atomic_symbols, convention)
        
        # Symmetrize the matrices
        pred_F_ao = 0.5 * (pred_F_ao + np.transpose(pred_F_ao, axes=(0, 2, 1)))
        
        true_ao_list.append(target_F_ao)
        pred_ao_list.append(pred_F_ao)

    true_ao_array = np.concatenate(true_ao_list, axis=0)
    pred_ao_array = np.concatenate(pred_ao_list, axis=0)

    mse = np.mean((pred_ao_array - true_ao_array)**2)
    mae = np.mean(np.abs(pred_ao_array - true_ao_array))
    
    print(f"{split_name:<10} | MSE: {mse:.8e} a.u. | MAE: {mae:.8e} a.u.")
    return true_ao_array, pred_ao_array

print("\nEvaluating all splits and transforming back to AO basis...")
print("-" * 65)
with torch.no_grad():
    train_true, train_pred = evaluate_loader(train_loader, "Train")
    val_true, val_pred     = evaluate_loader(val_loader, "Validation")
    test_true, test_pred   = evaluate_loader(test_loader, "Test")
print("-" * 65)

# Keep the test set predictions globally available for the plotting cells (Cells 7 & 8)
all_true_ao = test_true
all_pred_ao = test_pred

In [ ]:
# --- CELL 7: VISUALIZE A SINGLE PREDICTION ---
%matplotlib inline

def plot_fock_comparison(true_matrices, pred_matrices, sample_idx=0):
    if len(true_matrices) == 0:
        print("No data available! Please run Cell 6 first.")
        return

    F_true = true_matrices[sample_idx]
    F_pred = pred_matrices[sample_idx]
    F_err = F_pred - F_true
    
    fig, axes = plt.subplots(1, 3, figsize=(16, 5))
    vmax = max(np.max(np.abs(F_true)), np.max(np.abs(F_pred)))
    vmin = -vmax
    
    im0 = axes[0].matshow(F_true, cmap='bwr', vmin=vmin, vmax=vmax)
    axes[0].set_title("True Fock Matrix (AO Basis)", pad=15)
    fig.colorbar(im0, ax=axes[0], fraction=0.046, pad=0.04)
    
    im1 = axes[1].matshow(F_pred, cmap='bwr', vmin=vmin, vmax=vmax)
    axes[1].set_title("Predicted Fock Matrix (AO Basis)", pad=15)
    fig.colorbar(im1, ax=axes[1], fraction=0.046, pad=0.04)
    
    err_max = np.max(np.abs(F_err))
    # Add a tiny epsilon to err_max to prevent Matplotlib crashing if error is exactly 0.0
    err_max = err_max if err_max > 0 else 1e-10 
    
    im2 = axes[2].matshow(F_err, cmap='PiYG', vmin=-err_max, vmax=err_max)
    axes[2].set_title(f"Error (Max: {err_max:.5e})", pad=15)
    fig.colorbar(im2, ax=axes[2], fraction=0.046, pad=0.04)
    
    plt.tight_layout()
    
    # Bulletproof backup: Save to disk
    plt.savefig(os.path.join(ANALYSIS_DIR, "fock_matrix_comparison.png"), dpi=300, bbox_inches='tight', facecolor='white')
    print("Plot saved to 'fock_matrix_comparison.png'")
    
    plt.show()

# Visualize the first molecule in the test set
plot_fock_comparison(all_true_ao, all_pred_ao, sample_idx=0)

In [ ]:
# --- CELL 8: DATASET ERROR STATISTICS ---
def plot_error_statistics(true_matrices, pred_matrices):
    true_flat = true_matrices.flatten()
    pred_flat = pred_matrices.flatten()
    errors = pred_flat - true_flat
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    axes[0].scatter(true_flat, pred_flat, alpha=0.05, s=2, color='blue')
    
    min_val, max_val = np.min(true_flat), np.max(true_flat)
    axes[0].plot([min_val, max_val], [min_val, max_val], 'k--', lw=2, label="Ideal")
    axes[0].set_xlabel("True Matrix Elements")
    axes[0].set_ylabel("Predicted Matrix Elements")
    axes[0].set_title("Correlation (PhiSNet)")
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)
    
    axes[1].hist(errors, bins=150, color='red', alpha=0.7, edgecolor='black', linewidth=0.2)
    axes[1].set_xlabel("Error (Predicted - True)")
    axes[1].set_ylabel("Count")
    axes[1].set_title("Error Distribution")
    axes[1].axvline(0, color='k', linestyle='dashed', linewidth=2)
    axes[1].grid(True, alpha=0.3)
    
    p1, p99 = np.percentile(errors, [0.1, 99.9])
    axes[1].set_xlim(p1, p99)
    
    plt.tight_layout()
    plt.savefig(os.path.join(ANALYSIS_DIR, "dataset_error_statistics.png"), dpi=300, bbox_inches='tight', facecolor='white')
    plt.show()

plot_error_statistics(all_true_ao, all_pred_ao)

In [ ]:
# --- CELL 9: THE ROTATION EQUIVARIANCE TEST ---
import copy
import torch
import numpy as np
from scipy.spatial.transform import Rotation
import ase.db



def test_rotational_equivariance(model_path, db_path, convention):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = torch.load(model_path, map_location=device, weights_only=False).to(device)
    model.eval()
    print("Database path:", os.path.abspath(db_path))

    with ase.db.connect(db_path) as conn:
        print("Number of rows in database:", conn.count())
        row = next(conn.select())
        atoms_orig = row.toatoms()

    atoms_rot = copy.deepcopy(atoms_orig)
    rot_matrix = Rotation.from_euler('z', 45, degrees=True).as_matrix()
    atoms_rot.positions = atoms_rot.positions @ rot_matrix.T

    print("Testing PhiSNet...")
    print("Molecule rotated by 45 degrees around Z-axis.")

    with torch.no_grad():
        R_orig = torch.tensor(atoms_orig.positions, dtype=torch.float64, device=device).unsqueeze(0)
        R_rot = torch.tensor(atoms_rot.positions, dtype=torch.float64, device=device).unsqueeze(0)
        
        # FIXED: Pass ONLY the positions (R) to the raw model
        F_orig_lm = model(R_orig)['full_hamiltonian'].cpu().numpy().squeeze()
        F_rot_lm = model(R_rot)['full_hamiltonian'].cpu().numpy().squeeze()
        
        symbols = list(atoms_orig.symbols)
        
        F_orig = transform_hamiltonians_from_lm_to_ao(F_orig_lm[None, ...], symbols, convention)[0]
        F_rot = transform_hamiltonians_from_lm_to_ao(F_rot_lm[None, ...], symbols, convention)[0]
        
        # Symmetrize the matrices
        F_orig = 0.5 * (F_orig + F_orig.T)
        F_rot = 0.5 * (F_rot + F_rot.T)

    matrix_diff = np.max(np.abs(F_orig - F_rot))
    eig_orig = np.sort(np.linalg.eigvalsh(F_orig))
    eig_rot = np.sort(np.linalg.eigvalsh(F_rot))
    eigenvalue_diff = np.max(np.abs(eig_orig - eig_rot))

    print("-" * 40)
    print(f"Maximum difference in AO Matrix Elements: {matrix_diff:.6e}  <-- (EXPECTED TO BE LARGE! AO basis changes with rotation)")
    print(f"Maximum difference in Eigenvalues:        {eigenvalue_diff:.6e}  <-- (EXPECTED TO BE ZERO! Physics are invariant)")
    print("-" * 40)
    
    if matrix_diff > 1e-3 and eigenvalue_diff < 1e-4:
        print("✅ SUCCESS: The AO matrix changed properly under rotation, but the physical orbital energies remained perfectly stable.")

test_rotational_equivariance(
    os.path.join(OUT_DIR, f"phisnet_raw_{N_TRAIN_SAMPLES}.pt"), 
    os.path.join(DATA_DIR, "open.db"),     
    convention=convention                 
)
